In [ ]:
# ===============================
# 1. Instalacje
# ===============================
%pip install -U bitsandbytes sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 73.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
# ===============================
# 2. Importy
# ===============================
import torch
import numpy as np
import faiss
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer

In [ ]:
# ===============================
# 3. Model językowy – Qwen2.5
# ===============================
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# ===============================
# 4. Dokument źródłowy (związek biednych chłopów)
# ===============================
document = """
The Union of Poor Peasants (Ukrainian: Спілка бідних хліборобів), also known as the Peasant Group of Anarcho-Communists or the Huliaipole Anarchist Group, was an underground anarchist organization, operating in the years 1905–1908 in and around the area of Huliaipole in what is today Ukraine.
During the 1905 Revolution, a wave of anarchist activity erupted throughout Ukraine, with strike actions and soviets being organized in major cities, while militant groups such as the Black Banner carried out acts of terrorism against the Russian Empire.
But with the defeat of the revolution and the institution of reforms by the Russian prime minister Pyotr Stolypin, anarchists in the small southern Ukrainian town of Huliaipole began to consider a violent struggle against the Tsarist police to be their most immediate task.
In 1906, a Ukrainian Czech teacher called Voldemar Antoni began to share his anarchist political philosophy with former schoolmates in his hometown of Huliaipole, going on to establish a local anarcho-communist group: the Union of Poor Peasants.
The group attracted about a dozen core members, mostly from the region's peasantry, with new arrivals going through a probationary period of political education before becoming full members.
The group was swiftly joined by Oleksandr Semenyuta, a local peasant that had gone into clandestinity after committing draft evasion, who became the main supplier of the group's weaponry.
Between September 1906 and July 1908, the group carried out a series of armed robberies and assassinations.
On 18 September 1906, the group carried out its first robbery against a local merchant known as Pelshchiner, during which three armed members surrounded his house and forced him to hand over his cash and jewellery.
The group carried out a second robbery on 23 October 1906, stealing 500 rubles from a local nationalist poet Hryts’ko Kernerenko.
They spent this money on a second-hand hectograph, which they used to print their own leaflets, attacking Stolypin's reforms and railing against the kulaks.
On 26 November 1906, they carried out their third robbery against the local industrialist Mark Kerner, who was robbed of his cash and a silver ingot by seven group members, some of whom he reported had been shaking with nerves.
They sent him a letter two days later, regretting that they hadn't been able to steal even more of his money and threatening to bomb his home if he continued to inform the police of their activities.
Following the arrest of some of their members, including the young Nestor Makhno, the group decided to lie low for the remainder of the winter.
In August 1907, they regrouped and went to Haichul in order to carry out their fourth robbery against a merchant called Gurevich.
During the night, four armed group members broke into his house and demanded money, identifying themselves as anarcho-communists, but Gurevich's nephew cried out for help, forcing them to escape without any loot.
On 1 November  1907, they carried out another robbery against a post office cart, during which they killed a local police officer.
In compensation for his death, the group left 100 rubles of the stolen money with the officer's widow.
The police were initially unable to identify the assailants, but a prisoner in Katerynoslav named Nazarii Zuichenko informed them of the anarchists' identities: Voldemar Antoni had planned the attack and provided the weapons, while the attack itself had been carried out by Nestor Makhno, Anton Bondarenko and Prokip Semenyuta, who had also participated in other robberies.
Makhno himself was subsequently arrested on charges of murder and expropriation, but he was quickly released due to lack of evidence.
Despite the growing police investigation into their activities, the anarchists continued.
On 23 April 1908, Ivan Levadny and Naum Altgauzen led a raid on Bogodarovsk, robbing a merchant named Levin of his cash and gold.
On 26 May 1908, the gang attempted to rob another merchant but failed to steal any money, escaping after the merchant's daughter was wounded by their gunfire.
On 22 June 1908, the gang raided a state-owned liquor store in Novoselivka, during which a retail clerk was shot and killed.
After the group assassinated a police informant, the authorities began to actively hunt for members of the group.
Half of the group went underground and continued their activities in clandestinity, while other members were arrested.
Aided by informants, the police found out that the group had scheduled a meeting at the house of Ivan Levadny for 10 August 1908.
Ten policeman surrounded the house and a shoot-out ensued, during which the commanding officer of the police was killed.
The anarchists managed to escape into the night, but Prokip Semenyuta had been wounded in the leg and decided to shoot himself, in order to not slow down his fleeing comrades.
Makhno and Oleksandr Semenyuta subsequently attempted a number of attacks against the provincial governor, but these were all aborted and the group members fled in shoot-outs with the police.
Karachentsev, the chief of police in Huliaipole, responded by executing a series of arrests of the group members, despite still lacking sufficient evidence against them.
He tracked down a number of anarchists that were hiding in Katerynoslav, capturing Altgauzen, Lisovski, Levadny and Zuichenko.
During their interrogation, Levadny broke under pressure and informed the police of the group's entire history, while Altgauzen also confessed to participating in a number of robberies, for which he was later accused by Makhno of being an agent provocateur.
Following Zuichenko's confession, even more anarchists were arrested and the group was finally brought to trial.
The trial provided Karachentsev with the necessary information to arrest even more of Huliaipole's anarchists, including Nestor Makhno.
Although other group members ended up confessing, Makhno staunchly denied all charges against him.
But on 14 September 1908, a number of Makhno's notes were intercepted, one of which instructed Levadny not to incriminate their comrades and another which detailed plans to escape prison.
Testimony from witnesses, including Shevchenko's own brother, further incriminated the group members.
Having been identified as the group's leaders, Voldemar Antoni and Oleksandr Semenyuta fled to Belgium, where they plotted their next moves.
Despite many of the group's members recanting their confessions, claiming they were under duress, Zuichenko's subsequent confession incriminated all of them, with one of the group members being hanged on 30 June 1909 and another dying from typhus while imprisoned.
Transferred to Oleksandrivsk, the accused were kept in prison for a year while the prosecution continued their investigations.
With the threat of capital punishment hanging over their heads, a number of group members tried and failed to escape prison during the winter of 1909.
But Makhno only lasted two days before his recapture and Levadny died in a snowstorm.
It was around this same time that Oleksandr Semenyuta returned to Huliaipole from his Belgian exile.
Seeking revenge for his brother's death, Semenyuta assassinated Karachentsev after he left the local theatre.
With a bounty on his head, Semenyuta managed to elude capture for nearly a year, but he was surrounded at his house by police and shot himself in order to avoid capture.
In March 1910, the prisoners were transferred to Katerynoslav for their court-martial by the Odessa Military District.
Charged with expropriation, illegal association and illegal assembly, sixteen of the group's members were all found guilty and sentenced to death.
Although, after 52 days on death row, Makhno's sentence was commuted to life imprisonment due to his young age.
Makhno was subsequently transferred to Butyrskaya prison, where he came under the wing of Peter Arshinov and received a comprehensive education from him.
Makhno and his fellow prisoners were finally released during the February Revolution, following which Makhno returned to Huliaipole.
With most of their comrades in prison, dead or exiled, the few remaining members were only able to reconstitute the group in May 1916.
The reformed group subsequently shifted their focus toward propaganda, which they carried out up until the outbreak of the 1917 Revolution.
On the first day of the February Revolution, the group led a procession of black flags to the graves of their fallen comrades, including Prokip and Oleksandr Semenyuta.
On 24 March, Makhno arrived back in Huliaipole, where he was greeted by the group's surviving members.
Makhno struggled to persuade many of the group's members on his organizational tactics, as many of them still wished to focus on the distribution of propaganda, but he quickly won them over to his plan.
By the following week, he had established a broad-based Peasant Union and was elected as its chairman, enrolling almost all of the town's peasantry within the space of a few days.
When the local organ of the Russian Provisional Government, known as the Public Committee, held elections in early April, it was brought under the control of member and sympathisers of the Peasant Union.
This accelerated the pace of the revolution in Huliaipole, as local peasants and workers seized control of land and industry, establishing a network of agricultural communes throughout the region.
Inspired by the work of the Catalan pedagogue Francesc Ferrer i Guàrdia, the group also set about reforming the local education system.
Makhno himself raided the local police archives and discovered the identity of the police informants that had given away the group in 1908, eventually finding the informant known as Sharovsky and having him shot.
Within months, the Provisional Government representatives had been driven out of town and the participation of the peasantry in local affairs dramatically increased.
In August 1917, Makhno went to Katerynoslav, where he attended the Provincial Congress of Soviets and Peasant Unions as the delegate for Huliaipole.
The congress decided that the Peasant Unions of Katerynoslav province were to be reorganized into soviets.
In the wake of the October Revolution, the Huliaipole Soviet organized support for the Ukrainian People's Republic of Soviets, which brought southern Ukraine under "soviet power".
But following the ratification of the Treaty of Brest-Litovsk, Ukraine was invaded by the Central Powers.
By this time, tensions between the anarchists and Ukrainian nationalists had heightened.
When one member of the Ukrainian Socialist-Revolutionary Party threatened the anarchists with retribution, the anarchist group responded by murdering him.
Makhno attempted to heal the divisions by setting up a joint commission of both anarchists and nationalists, but this alienated younger group members, who saw it as a compromise with "counter-revolutionaries".
In response, Makhno proposed that the group form a revolutionary vanguard and take a leading role in combatting the counter-revolution.
They subsequently organized peasant detachments to resist the invasion, but after they were dispatched, the Union of Poor Peasants was deposed in a coup by Ukrainian nationalists and the town fell under the control of the Austro-Hungarian Army.
The coup was aided by one of the group's former members, Lev Schneider, who had joined the nationalist cause.
In April 1918, Ukrainian anarchist partisans regrouped in Taganrog, where they held a conference to discuss the situation and how they could respond.
Makhno urged the Union of Poor Peasants that had remained in Huliaipole to resist the occupation and to once again raise their flag, which read "Always with the oppressed against the oppressors."
The partisans also planned to themselves return to Huliaipole in July 1918, in order to ignite an insurrection against the Ukrainian State.
Upon their return, they became the nucleus for the Revolutionary Insurgent Army of Ukraine, which brought much of southern Ukraine under anarchist control throughout the Ukrainian War of Independence.
Many members of the Union of Poor Peasants went over to the insurgent military, in order to defend Huliaipole against any prospective rulers.
One of the group's members, Abram Budanov, founded a cultural organisation for the Insurgent Army and began the publication of anarchist propaganda.
In October 1918, the Union of Poor Peasants regrouped in Huliaipole and expanded their influence, establishing branches in Dibrivka and Pokrovske.
By December, they had reopened an anarchist club in the town and published a series of leaflets agitating the peasantry to fight against the emergent White movement.
The group also restarted the establishment of communes, eventually dropping propaganda work in favor of either organizational or military tasks.
When an alliance with the Bolsheviks was first proposed in January 1919, it was opposed by the Union of Poor Peasants, with one of their delegates to an anarchist congress in Yelyzavethrad displaying marked anti-Bolshevism.
That same month, the Union was joined in Huliaipole by another anarchist organization: the Nabat.
Relations between the two were initially cordial, as both had disagreed with the Makhnovist-Bolshevik alliance.
But a dispute over the territory of the Makhnovshchina caused a split between them, as the Huliaipole anarchists insisted on controlling their home region, above all else.
By the summer of 1919, definitive anarchist control of Huliaipole had again been lost, this time to the White movement.
The Union of Poor Peasants then started to disappear from the historical record, as it was no longer able to effectively and openly work under occupation by the Armed Forces of South Russia.
"""

In [ ]:
def chunk_text_manual(text, n, overlap=2):
    sentences = []
    sentence = ""
    for char in text:
        sentence += char
        if char in ".!?,":
            sentence = sentence.strip()
            if sentence:
                sentences.append(sentence)
            sentence = ""
    if sentence.strip():
        sentences.append(sentence.strip())

    chunks = []
    i = 0
    while i < len(sentences):
        chunk = " ".join(sentences[i:i + n])
        chunks.append(chunk)
        if i + n >= len(sentences):
            break
        i += n - overlap

    return chunks


chunks = chunk_text_manual(document, n=7)

In [ ]:
# ===============================
# 6. Embeddingi
# ===============================
embedder = SentenceTransformer("intfloat/multilingual-e5-base")

chunk_embeddings = embedder.encode(
    chunks,
    normalize_embeddings=True
)

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(chunk_embeddings))

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
# ===============================
# 7. Retrieval
# ===============================
def retrieve_context(query, k):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_emb), k)
    return "\n\n".join([chunks[i] for i in indices[0]])

In [ ]:
# ===============================
# 8. Funkcja ask_bot
# ===============================
def ask_bot(question):
    #próba z małym k
    context = retrieve_context(question, k=4)

    # prompt RAG
    def generate_answer(q, ctx):
        prompt = f"""
You are a precise QA system.

Rules:
- If the answer requires reasoning or inference, make a logical conclusion.
- If the answer IS NOT EXPLICITLY in the context, respond: "No information in the document."
- Use ONLY the context as your source of knowledge.
- Output ONLY ONE clear, natural easy to read sentence that answers the question.
- Do NOT repeat the question.
- Do not add any unnecessary information or commentary.

CONTEXT:
{ctx}

QUESTION:
{q}

ANSWER:
"""
        output = generator(
            prompt,
            max_new_tokens=60,
            temperature=0.2,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            return_full_text=False
        )[0]["generated_text"]

        ans = output.strip().split(".")[0] + "."
        ans = ans.split("Human:")[0].strip()
        ans = ans.split("Assistant:")[0].strip()

        if "No information in the document" in ans:
            return "No information in the document."
        else:
            return ans.rstrip(".") + "."

    answer = generate_answer(question, context)
    if answer == "No information in the document.":
        context = retrieve_context(question, k=7)
        answer = generate_answer(question, context)

    # print pytania i kontekstu
    print("-" * 80)
    print("PYTANIE:")
    print(question)
    print("-" * 40)
    print("KONTEKST:")
    print(context)
    print("-" * 40)

    # print odpowiedzi
    print("ODPOWIEDŹ:")
    print(answer)
    print("-" * 80)

    return

In [ ]:
# ===============================
# 9. Testy
# ===============================
ask_bot("What was the union of poor peasants?")
ask_bot("How did the failure of the 1905 Revolution influence the Union of Poor Peasants’ strategy?")
ask_bot("Who was the main supplier of the group's weaponry")
ask_bot("In what kind of activities the memebers of the union of the poor peasants were engaged in?")
ask_bot("What were the other names of the union of poor peasants?")
ask_bot("What happened to the union of poor peasants?")
ask_bot("Why were they poor?")
ask_bot("Why did the relationship with the anarchist group Nabat eventually deteriorate?")
ask_bot("At what temperature does the water boil?")
ask_bot("What's the meaning of life?")

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------------------------------------
PYTANIE:
What was the union of poor peasants?
----------------------------------------
KONTEKST:
The Union of Poor Peasants (Ukrainian: Спілка бідних хліборобів), also known as the Peasant Group of Anarcho-Communists or the Huliaipole Anarchist Group, was an underground anarchist organization, operating in the years 1905–1908 in and around the area of Huliaipole in what is today Ukraine. During the 1905 Revolution, a wave of anarchist activity erupted throughout Ukraine, with strike actions and soviets being organized in major cities,

the Union of Poor Peasants was deposed in a coup by Ukrainian nationalists and the town fell under the control of the Austro-Hungarian Army. The coup was aided by one of the group's former members, Lev Schneider, who had joined the nationalist cause. In April 1918, Ukrainian anarchist partisans regrouped in Taganrog, where they held a conference to discuss the situation and 

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------------------------------------
PYTANIE:
How did the failure of the 1905 Revolution influence the Union of Poor Peasants’ strategy?
----------------------------------------
KONTEKST:
above all else. By the summer of 1919, definitive anarchist control of Huliaipole had again been lost, this time to the White movement. The Union of Poor Peasants then started to disappear from the historical record, as it was no longer able to effectively and openly work under occupation by the Armed Forces of South Russia.

The Union of Poor Peasants (Ukrainian: Спілка бідних хліборобів), also known as the Peasant Group of Anarcho-Communists or the Huliaipole Anarchist Group, was an underground anarchist organization, operating in the years 1905–1908 in and around the area of Huliaipole in what is today Ukraine. During the 1905 Revolution, a wave of anarchist activity erupted throughout Ukraine, with strike actions and soviets being organized in major citi

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

--------------------------------------------------------------------------------
PYTANIE:
Why did the relationship with the anarchist group Nabat eventually deteriorate?
----------------------------------------
KONTEKST:
the Union was joined in Huliaipole by another anarchist organization: the Nabat. Relations between the two were initially cordial, as both had disagreed with the Makhnovist-Bolshevik alliance. But a dispute over the territory of the Makhnovshchina caused a split between them, as the Huliaipole anarchists insisted on controlling their home region, above all else. By the summer of 1919,

eventually dropping propaganda work in favor of either organizational or military tasks. When an alliance with the Bolsheviks was first proposed in January 1919, it was opposed by the Union of Poor Peasants, with one of their delegates to an anarchist congress in Yelyzavethrad displaying marked anti-Bolshevism. That same month, the Union was joined in Huliaipole by another anarchist orga

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------------------------------------
PYTANIE:
At what temperature does the water boil?
----------------------------------------
KONTEKST:
On 22 June 1908, the gang raided a state-owned liquor store in Novoselivka, during which a retail clerk was shot and killed. After the group assassinated a police informant, the authorities began to actively hunt for members of the group. Half of the group went underground and continued their activities in clandestinity, while other members were arrested.

Zuichenko's subsequent confession incriminated all of them, with one of the group members being hanged on 30 June 1909 and another dying from typhus while imprisoned. Transferred to Oleksandrivsk, the accused were kept in prison for a year while the prosecution continued their investigations. With the threat of capital punishment hanging over their heads, a number of group members tried and failed to escape prison during the winter of 1909. But Makhno onl

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------------------------------------
PYTANIE:
What's the meaning of life?
----------------------------------------
KONTEKST:
Charged with expropriation, illegal association and illegal assembly, sixteen of the group's members were all found guilty and sentenced to death. Although, after 52 days on death row, Makhno's sentence was commuted to life imprisonment due to his young age. Makhno was subsequently transferred to Butyrskaya prison,

where he was greeted by the group's surviving members. Makhno struggled to persuade many of the group's members on his organizational tactics, as many of them still wished to focus on the distribution of propaganda, but he quickly won them over to his plan. By the following week, he had established a broad-based Peasant Union and was elected as its chairman, enrolling almost all of the town's peasantry within the space of a few days.

a number of Makhno's notes were intercepted, one of which instructed Levad